# Causal attention over packed documents

LLM pre-training packs several short documents into one fixed-length row to avoid padding. A plain causal mask would then let a document attend to the end of the previous one. Build the **document-aware causal mask** and use it.

Write `packed_causal_attention(q, k, v, doc_ids)`:

- `q`, `k`, `v`: float64 `(B, nh, T, dh)`, already split into heads.
- `doc_ids`: int64 `(B, T)`. Each token's document id; `-1` marks padding. Ids are only compared for equality within a row.

Query $i$ may attend to key $j$ (same row) iff all of:

1. $j \le i$ (causal),
2. `doc_ids[b, i] == doc_ids[b, j]` (same document),
3. `doc_ids[b, j] != -1` (not padding).

Scores are $q_i \cdot k_j / \sqrt{d_h}$, softmaxed over the allowed keys; output row $i$ is the weighted sum of `v`. Every real query can attend to at least itself. Padding queries output zeros.

Return a tuple `(out, pos)`:

- `out`: `(B, nh, T, dh)`.
- `pos`: int64 `(B, T)`, each token's position **within its document** (0 for a document's first token), the position ids you'd feed to RoPE. Padding gets `-1`.

Use tensor ops, no Python loops over batch, heads or positions.

This problem runs in **Colab** (PyTorch isn't available in the browser). The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch

def packed_causal_attention(q, k, v, doc_ids):
    """Return (out (B, nh, T, dh), pos (B, T) int64)."""
    T = q.shape[-2]
    allowed = torch.ones(T, T, dtype=torch.bool).tril()           # TODO: documents and padding
    s = (q @ k.transpose(-2, -1)) / q.shape[-1] ** 0.5
    s = s.masked_fill(~allowed, float("-inf"))
    pos = torch.arange(T).expand_as(doc_ids)                       # TODO: restart at each document
    return torch.softmax(s, dim=-1) @ v, pos


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[[[0.0], [0.0], [0.0], [0.0]]]], "dtype": "float64"}, {"__nd__": [[[[0.0], [0.0], [0.0], [0.0]]]], "dtype": "float64"}, {"__nd__": [[[[1.0], [2.0], [3.0], [4.0]]]], "dtype": "float64"}, {"__nd__": [[0, 0, 1, 1]], "dtype": "int64"}], "e": {"__tup__": [{"__nd__": [[[[1.0], [1.5], [3.0], [3.5]]]], "dtype": "float64"}, {"__nd__": [[0, 1, 0, 1]], "dtype": "int64"}]}}, {"a": [{"__nd__": [[[[1.0], [0.0], [2.0]]]], "dtype": "float64"}, {"__nd__": [[[[1.0], [1.0], [1.0]]]], "dtype": "float64"}, {"__nd__": [[[[3.0], [5.0], [7.0]]]], "dtype": "float64"}, {"__nd__": [[4, 4, -1]], "dtype": "int64"}], "e": {"__tup__": [{"__nd__": [[[[3.0], [4.0], [0.0]]]], "dtype": "float64"}, {"__nd__": [[0, 1, -1]], "dtype": "int64"}]}}, {"a": [{"__nd__": [[[[-0.39530128858657, 0.2639148850157296, 0.6071282687955677], [-0.9721597997272025, 0.7676642531398922, 0.25505812177433956], [0.7829798706300901, 0.27241823407236765, 1.1620081255305676], [-0.9377563787668719, 1.776099733817743, 1.2023898441352874], [-0.5999578484570015, 0.6601663130603562, 0.44476738448754977], [-1.7457635656172505, 0.5952162456256633, -0.5853991209052885]], [[-0.25001353358513434, -0.6023864391110364, -0.4280840067640782], [0.07173470176248886, 0.09671912405619901, -1.5591518413922796], [-0.26870603364939344, -1.3448111474077982, -1.2707778590591046], [-0.346953600730839, 0.85530027587691, 0.6308565040330028], [-0.6046580474158655, -0.7104208239983351, -0.8274098970826956], [0.14274021925565974, 0.936650038779724, 0.01804985703847165]]], [[[0.6927666696591163, 0.3154090546513482, 1.5041856647552274], [-2.0066907365561706, -2.1310340831936396, -0.19843957784596147], [0.6365395187683154, -0.410549281758414, 0.42578068000505975], [-1.173544797067956, -1.2593832343358193, -0.7165758868835466], [0.8413227181749481, 0.696358369654102, -0.713252712154986], [0.18111936994597383, -0.6475708354871698, 0.020059810497343504]], [[-1.1142963805907677, -0.04655434751901477, -0.735753934980961], [-0.04435950785702231, -0.6520426060122302, -0.8217414802839252], [0.11343128787139657, -0.33779536082531136, 0.021671175782686587], [-0.419085181552037, 0.6137428973723346, -0.8156847530495072], [0.7687510449895725, 1.331974309444846, -0.26888770649361554], [1.7452430767280316, 2.029698196865837, 0.4430750030154757]]]], "dtype": "float64"}, {"__nd__": [[[[-1.3576051491193484, -1.6156939484528159, -0.1757786841969012], [1.7759298264822208, 0.574549849965791, 0.7580309194582652], [-0.2509133097904481, -0.7441468728010533, 0.37192110436980064], [-1.1087787163755936, -0.19694438865759697, 0.511152295497356], [0.03593415340311622, 0.5245642290612524, -0.4459765954576025], [0.5981798358088446, 0.4046503807009621, -1.6497062609235813]], [[-1.0422629538107075, -0.723476319045549, 0.6555863140819842], [0.3020801614586051, 0.2829882917455624, 0.17293243462011323], [0.6364764540263609, 0.9453478418566444, -0.55471618453936], [-1.5909121646919766, 0.6673244554529724, -0.20942920166475995], [0.04880537673064059, -0.1738052649063213, -1.1125682987307608], [-1.5106513146782443, 0.9642801867742248, 1.0930925298253509]]], [[[0.9312396637512143, 0.20823464575150918, 0.2685207840712548], [-0.4189296731440312, 0.068012235199191, -0.402836079808958], [-0.5436861645760849, 0.20520316656139895, 1.698100442046864], [-1.8835414686235434, 0.601511318239193, 0.6756512596372449], [-1.214406143769667, 0.31344994259797077, 0.10157496332371774], [-0.4639827216892454, -0.4963243116217951, -1.4061731881369968]], [[1.707177778357255, 2.3475675133295377, 1.5863758938692298], [-0.28779325388649285, 0.2856819811333672, 0.3772081316205152], [-0.8129329524007229, -0.2585133595997185, 0.5332454085822927], [-0.42162294678862666, -1.7597561158297332, -1.2680186486126208], [0.25498837659016704, -0.3370613965236977, -0.2982936118044847], [1.3559778672193719, 0.08478662296430321, -0.7174362329016395]]]], "dtype": "float64"}, {"__nd__": [[[[-0.06496331660890184, 2.1579108218895597, -1.1264636491083797], [1.786948338595363, 1.441594990780037, -0.1600953058439699], [-0.3358894071127189, 1.4188083086913799, 0.24397017291123366], [-0.5308329934263428, 1.8227617267578917, 0.1744923763100515], [0.33214588311318144, 0.03851839966105425, -0.3013107249042945], [0.12420019946696831, -0.3700959464536447, 0.5075311129071819]], [[1.0554538371570383, 1.085198198172033, 1.1913930981684038], [1.4954686333505183, 0.7610350163668245, -0.6044836257551762], [0.9621555437726979, -1.649206803175677, -0.7133713481582212], [0.28996866572309277, -1.6823936753767161, 1.4718807814018096], [-1.017348199619935, -1.4978022370187938, -0.7614302141268862], [-0.7640095664566192, -0.026283924899902982, -0.15793461410276913]]], [[[1.0310812013462103, -0.16643926260653358, 0.17555119355907506], [-0.9899791199757738, 0.497248770774718, -0.7749791139275669], [-1.9482928215392434, 1.3038350841125743, -1.0564318047764338], [-1.089701986610318, -0.8766761551128621, 0.6511177215972471], [-1.5234259916416433, -0.3981399491942163, 0.5768841319424516], [1.8969394746706036, -0.28968507982965547, 0.055826508337780724]], [[-0.8672504362875534, 1.9812905890587145, -0.14056637199013602], [-0.06256305126712666, 0.14671833729993328, -0.9161278230111044], [-0.01019480220174812, -1.2612591445422452, 1.4888421008879589], [-0.1494730398924859, -0.16486459210158966, 1.1681957582785087], [-1.2158558848220218, 1.401256351044982, 1.5222818221033567], [-0.7692632036460255, 0.690747539424684, -0.09757600098494713]]]], "dtype": "float64"}, {"__nd__": [[0, 0, 0, 1, 1, 2], [5, 5, 5, 5, 5, 5]], "dtype": "int64"}], "e": {"__tup__": [{"__nd__": [[[[-0.06496331660890184, 2.1579108218895597, -1.1264636491083797], [0.5700022925262811, 1.9123073576230825, -0.795124590894956], [1.2635905626253332, 1.4863411514588236, -0.15070337537976705], [-0.5308329934263428, 1.8227617267578917, 0.1744923763100515], [-0.17770530247639957, 1.0926561085102433, -0.02020445236829088], [0.12420019946696831, -0.3700959464536447, 0.5075311129071819]], [[1.0554538371570383, 1.085198198172033, 1.1913930981684038], [1.3341103851670983, 0.8799092078579942, 0.054084051133280034], [1.1461168327457287, 0.31471680880316305, 0.24760492991855493], [0.28996866572309277, -1.6823936753767161, 1.4718807814018096], [-0.43013556195049235, -1.5807158965017305, 0.2417148344734526], [-0.7640095664566192, -0.026283924899902982, -0.15793461410276913]]], [[[1.0310812013462103, -0.16643926260653358, 0.17555119355907506], [-0.7066427505984336, 0.4042050571390812, -0.6417224251167012], [-0.46958538189915877, 0.4986858554430888, -0.4709352388869832], [-0.9755400377795722, -0.012916085360941368, -0.1400686110536406], [-0.3947758190427344, 0.05073863679528615, -0.08584203193337635], [-0.25434578906712696, 0.04364632056733051, -0.09913939151884986]], [[-0.8672504362875534, 1.9812905890587145, -0.14056637199013602], [-0.22161897024274918, 0.5093431042126926, -0.762828987201623], [-0.2612596210825624, 0.1130653712331121, 0.20271002497958163], [-0.23432582073117653, 0.08337942362508798, 0.41994170607328113], [-0.7775275449664018, 1.6170338928291645, 0.007211805148538123], [-0.8573281524264365, 1.9188278762498931, -0.13116257062496675]]]], "dtype": "float64"}, {"__nd__": [[0, 1, 2, 0, 1, 0], [0, 1, 2, 3, 4, 5]], "dtype": "int64"}]}}, {"a": [{"__nd__": [[[[0.8827048205744942, 0.04195950390747259, -0.9001685960221104, 0.6281111109553829], [-0.21847528930025628, 0.1792444521332749, 0.9764409903166166, 1.2152746518399424], [0.6291961792584582, -1.4211155964990294, 0.6944321886200565, -0.3005015865564779], [1.0722490582691877, -0.19584167337101632, 1.4259985863620415, 0.5140710669046085], [6.793543485522398e-05, 0.2932726544441482, -1.7639750667525753, 0.5738072281730505]], [[0.11494091340878991, -1.0967977188048088, 0.4070016227208897, 0.3192577348176515], [0.3576206242975661, -0.11539345708878417, 0.7885177058202848, 0.8014461611354436], [0.09359147673978564, -0.06111520178016605, -0.4537959833660088, -0.5950699624384738], [0.6117625185880927, -0.5458992774564005, -1.4122232079667902, 0.24595977005172406], [1.1074164771267676, -0.27360636966135565, 0.8682156166641356, 0.48149846923416645]], [[-1.0081942468903757, 0.32246976768228164, -0.10889124296772344, 0.682223880607984], [-2.7856248635736574, 0.8312948280788758, 0.7414124616728882, -0.9247534906727276], [1.0697388192933963, -0.15130722249107023, 0.1845022021434689, 0.8662132063215051], [1.3465386087585518, 0.17360481455514865, 0.688291577557406, 0.4462666827414231], [-0.7248194388554418, 1.6449662232519746, 0.7813403082745828, 0.156700131611198]]]], "dtype": "float64"}, {"__nd__": [[[[1.083947940445542, 0.5539845188708896, -0.5421837845932549, 1.0166933918112726], [0.0784588327669846, 0.6362290891024897, 0.9726846896837326, 0.9654582584990465], [1.8991630189112358, 0.21829731730818147, -0.1466890000518128, -0.7031407553053078], [-0.6027897264430842, 0.6930890952459619, 0.4513877187669522, -0.8346272758764623], [-0.5825162357253724, -0.4084715404896509, 1.5894499115276843, -0.8972784619763616]], [[0.39727334202109627, -1.5898148267160814, -0.19121954630797752, 1.42350731339307], [-1.2102336016678912, 1.1903412349091838, -1.8462027198509605, 0.09528207839182529], [-1.063329287546252, 0.01901480233312343, -0.20140987821544276, 0.8510774053440736], [0.3089255698567089, 0.2063516130471289, -0.5036148267046537, 0.4865593554618768], [-0.7586924235367061, -0.6019229252180535, -0.5874203475439159, 0.6054588380533384]], [[2.298340994932344, -0.025761464349973624, 1.9574046479745844, -0.9767245878111889], [-0.5579163007463475, 0.5591204937058211, 0.6481375284198518, 0.8010694444716239], [1.6163401231181, -0.16196657693915434, 1.124964555390539, 0.290257705909035], [-2.780587856639197, -0.9478462610353828, -0.48086995513114983, 0.1566597759404357], [0.5753792545869604, -0.228369719056136, 1.3561969295906768, -0.9229505698673857]]]], "dtype": "float64"}, {"__nd__": [[[[0.021755413254581496, 0.9650007879542244, -1.075555449216515, 1.3008931319604966], [0.9983919324818832, -0.11492114694347667, 0.05099400573225651, -0.040401233830665594], [0.06839208673797367, 1.2911578682388358, 0.5613875882114832, -1.3276869360720291], [-1.4929348133812654, 0.6474104502169915, -2.5776150647263334, -1.5289541262487212], [-0.5708026153496051, 0.016263792020536745, 1.3869352166445277, 0.21359480860371988]], [[3.543037704854227, 0.2497892779874184, 0.8905138867723432, 1.0490736733613784], [-0.07181970439759625, 0.9846828853544083, 0.6284646320874582, 0.26901647976402876], [-0.5988590612920697, -0.39397434228861405, 0.8278662777624761, -0.0032894717200770427], [0.43514842476537907, 0.43553125634532486, 0.31011626601262127, 0.04715470467760803], [0.619000989644874, 0.8614473173742426, 1.4039926408672414, 0.5322123238340821]], [[-0.5289781911722888, -0.6422201331945767, 0.6691192219994475, -1.4880560336530309], [-0.485151456671021, 1.070965457225669, -0.29628428165655657, -0.6187967335494434], [1.2623355436372012, -2.613171741623092, -0.04304455344124896, -0.42336415922630627], [-0.4543963707910916, 0.9381310352439323, 1.5284555439125136, 1.1630782120495018], [0.6534404883193268, 0.35796645765382007, -0.20916732233746632, 0.03651710553388239]]]], "dtype": "float64"}, {"__nd__": [[0, 1, 2, 3, 4]], "dtype": "int64"}], "e": {"__tup__": [{"__nd__": [[[[0.021755413254581496, 0.9650007879542244, -1.075555449216515, 1.3008931319604966], [0.9983919324818832, -0.11492114694347667, 0.05099400573225651, -0.040401233830665594], [0.06839208673797367, 1.2911578682388358, 0.5613875882114832, -1.3276869360720291], [-1.4929348133812654, 0.6474104502169915, -2.5776150647263334, -1.5289541262487212], [-0.5708026153496051, 0.016263792020536745, 1.3869352166445277, 0.21359480860371988]], [[3.543037704854227, 0.2497892779874184, 0.8905138867723432, 1.0490736733613784], [-0.07181970439759625, 0.9846828853544083, 0.6284646320874582, 0.26901647976402876], [-0.5988590612920697, -0.39397434228861405, 0.8278662777624761, -0.0032894717200770427], [0.43514842476537907, 0.43553125634532486, 0.31011626601262127, 0.04715470467760803], [0.619000989644874, 0.8614473173742426, 1.4039926408672414, 0.5322123238340821]], [[-0.5289781911722888, -0.6422201331945767, 0.6691192219994475, -1.4880560336530309], [-0.485151456671021, 1.070965457225669, -0.29628428165655657, -0.6187967335494434], [1.2623355436372012, -2.613171741623092, -0.04304455344124896, -0.42336415922630627], [-0.4543963707910916, 0.9381310352439323, 1.5284555439125136, 1.1630782120495018], [0.6534404883193268, 0.35796645765382007, -0.20916732233746632, 0.03651710553388239]]]], "dtype": "float64"}, {"__nd__": [[0, 0, 0, 0, 0]], "dtype": "int64"}]}}, {"a": [{"__nd__": [[[[-1.5834389250723568, -0.48761777182063926], [0.6277426891564911, -0.08652953384541183], [0.8559592052907321, -1.5165857769098154], [-0.5270207440038213, -0.9652652774319342], [1.0826169676052821, -0.22852941708183167], [-0.2593539245473229, -1.1851441572187045]]], [[[-0.4682269993858051, -0.9727950999803793], [-0.5356855045541156, 2.197944909271107], [-1.1169816122467204, -0.2275369056493768], [-0.8591797430809386, 1.3312807167846197], [-0.9384475978636716, -0.8350302356380308], [0.3929015548636508, 1.6792041514919482]]]], "dtype": "float64"}, {"__nd__": [[[[0.25836071689685364, -0.5410174522914828], [-2.520631018613691, -3.584464584498713], [-0.05720273081033771, 0.04819969234298533], [1.199306119249865, 0.4874436319125008], [-0.2391422732717059, 1.2977532333043655], [1.7428299371830858, -0.6324559559158519]]], [[[0.9449775759912037, 1.2103668614985865], [-0.7433684940829819, 0.5822667371140716], [-0.013637988511873488, 1.1899148918032372], [1.1226704142190722, -2.099175415769622], [-0.20800226228135257, 1.3876175613040804], [0.5265130959327202, -1.008304367161351]]]], "dtype": "float64"}, {"__nd__": [[[[-0.4625836415532583, -1.05281223910202], [0.34350706919125823, -1.2836406875341577], [1.2366054731858596, 0.863166544102697], [1.6956836512626114, 1.0935087058739794], [1.038077645954742, 0.9365491260557999], [0.5971190350755541, -1.9021104848480053]]], [[[0.5834382685662215, -0.7524120456860055], [-2.3580312569655235, 1.6068954466233776], [0.0261579816083452, -1.58937290345735], [0.3385200945492352, 0.4669429496197702], [0.22740769110188155, 1.5045219792716589], [-1.0760059601468912, 0.932050450373597]]]], "dtype": "float64"}, {"__nd__": [[7, 7, 3, 3, -1, -1], [-1, -1, -1, -1, -1, -1]], "dtype": "int64"}], "e": {"__tup__": [{"__nd__": [[[[-0.4625836415532583, -1.05281223910202], [-0.25321063188344484, -1.1127673363652208], [1.2366054731858596, 0.863166544102697], [1.3820873926824355, 0.9361619976785244], [0.0, 0.0], [0.0, 0.0]]], [[[0.0, 0.0], [0.0, 0.0], [0.0, 0.0], [0.0, 0.0], [0.0, 0.0], [0.0, 0.0]]]], "dtype": "float64"}, {"__nd__": [[0, 1, 0, 1, -1, -1], [-1, -1, -1, -1, -1, -1]], "dtype": "int64"}]}}, {"a": [{"__nd__": [[[[-0.9419798968979491, 0.5046000993693522], [0.6818916000021091, -0.4501282150583147], [2.553538163324132, -1.7746614690751428], [-1.4509872313625893, -0.4346076495334098], [-0.8626020675591859, 1.1667429918297663], [-0.5529672845815184, 1.3640046319118677], [-0.4121207900177814, 0.8367991478042597]], [[0.8300655081331314, 0.146605675933901], [0.7459577304500747, 0.3809039562758178], [2.1447653177968777, 2.4106515492974823], [-0.30921291727724054, 0.5258227658197887], [1.1899629680001662, 0.6051384647338347], [2.0072734778324866, 2.6779760585134786], [-1.9439350275835687, -0.5113289297867515]]]], "dtype": "float64"}, {"__nd__": [[[[1.3721484390789787, 0.5701846480703028], [-0.6720284032871408, -0.19949554983507], [0.1727953592835805, 1.6986084401054489], [-0.9121340827576457, 0.25490677825520874], [1.5241605550241053, 1.0937745243698704], [-0.09314205986075712, -0.23793413086278203], [-1.7494406854297933, -1.6729521778675704]], [[-0.8457815746924643, 0.8755425083978499], [0.8541547172219084, 1.3367604909133983], [0.9500565669378069, 0.7979133300901686], [0.20529170332748603, -0.7635186916406551], [1.2554240331280655, -0.3438671051240644], [-1.783871765990227, 0.4823305265031308], [0.12101549850619843, 2.4667028231800514]]]], "dtype": "float64"}, {"__nd__": [[[[-0.23714642821145163, 0.38016489382988733], [-1.12817795226139, 0.7105526317732384], [0.2889063783008983, -0.7916283006828269], [0.26421032693377206, -1.2588024784373084], [0.5971057381299691, -0.4445310619618362], [0.3746279375311829, -0.8981724941457758], [-0.7606667761236254, -0.15165750206219078]], [[0.846519787982925, 0.14029012424086149], [-2.373347930114152, 0.679352534929921], [-0.10356658136067173, 0.6440686888741336], [1.1608427705895605, 0.8357692788865869], [-0.7258838536040022, -0.3577231902540261], [0.5077568908328721, 1.0190932544083882], [-0.7686432792448947, -0.6319220577510385]]]], "dtype": "float64"}, {"__nd__": [[0, 0, 1, 1, 1, 0, 0]], "dtype": "int64"}], "e": {"__tup__": [{"__nd__": [[[[-0.23714642821145163, 0.38016489382988733], [-0.5248246067365048, 0.48683378274478606], [0.2889063783008983, -0.7916283006828269], [0.26851002784361516, -1.1774652149499156], [0.3252981092255977, -0.8901309127242495], [-0.3922964546685761, 0.13988077621167316], [-0.4469584883590758, 0.06502771771959663]], [[0.846519787982925, 0.14029012424086149], [-1.5205545660629365, 0.5365799470806394], [-0.10356658136067173, 0.6440686888741336], [0.3984828457258678, 0.7201857875332489], [-0.1748049535854832, 0.2850553217679854], [-2.244832562549803, 0.6615369088234175], [0.4916901010328296, 0.8063114261832474]]]], "dtype": "float64"}, {"__nd__": [[0, 1, 0, 1, 2, 2, 3]], "dtype": "int64"}]}}, {"a": [{"__nd__": [[[[-0.9879582319720063, 0.41138998064590165, -0.7186739256029496, 2.3346049954224726], [0.8425583759574132, 1.195311401854755, 0.6160955250669505, -0.5910398391511055], [-2.5523479950694, 1.4707382174243644, 0.3168754657511549, -1.89251174577699], [2.857417308398872, 0.759595883493327, 1.6143240839550215, -0.23962127741574016], [-1.773524406164664, -0.7770264193549921, 0.8052572197807112, -2.1145687683611256], [-0.5269166797239874, 1.1634153363788853, 1.0872115590057068, 0.08793649372992199], [-2.569815007014502, -0.2549367243839244, 1.303741995108828, 0.8854002368207518], [-0.38818820894617545, 0.3908516033786845, 0.5884590854717898, -0.3863752266147558]], [[-0.7310280445011604, -0.6946631583986302, -0.8359198746921428, -0.0668886643421378], [-0.030821338920872098, 0.5965975803542072, 0.6009596523423841, -0.12646372351796983], [-2.1368574949812524, 0.5723321309744546, -0.8873356654487232, 1.7576424746370187], [-0.2896389613543849, 2.2650321029177523, 0.6562109503422947, 1.6972239036574002], [0.21481393203045368, 0.5325136636864569, -0.40745078122867684, -0.45582070811583525], [-0.4552024117544363, 0.7678586759290762, -1.111286646322974, 1.0416642592520229], [-0.24414706728703633, 0.6470335250769098, -0.9680948851454563, 0.07737495160479457], [-1.694391985059433, -0.631376215399936, -0.18627436908099249, -1.0059627847383756]]], [[[-0.22322685693590713, 0.12271004525398393, 0.14007771582352174, -2.0414699690841194], [-0.3008371133955535, -0.2119577842665349, -0.8498441404014584, -0.9611077560967894], [1.9240530983498008, 0.8394057773816513, 0.9947871939776144, 1.195285382059803], [-0.9367231568748363, -0.2560857626918455, -1.7697302468301153, 0.18869368941719586], [-1.9147093004865956, -0.7230841830650089, 0.744665933513062, 0.5401005529965468], [-0.9814415212539587, -0.8986512872504236, 1.3031267081215894, 0.445946892760851], [-0.5042356567469672, 0.1442878072600871, 0.12388610467600215, -1.0764393343449736], [-0.5541423916282453, -2.105225520016432, 0.238083436191829, 0.08277342984905878]], [[1.0373445120611795, 1.029620992798984, -0.45418041381695634, 1.600964614199409], [-0.7315324396221344, 0.9397301517648157, -0.5193395909052296, -0.6208443329521166], [-1.2433080161016339, -0.062287563022193584, 2.2565703190412596, 0.15589439387235776], [0.4730592957333231, -0.33943750041431664, 0.5049340947463096, 0.6251394508608439], [0.47841517359771607, 1.376749516495455, -0.02247784141705673, 0.421090834154706], [1.5518391975153933, 0.6295466715635412, -0.744873033074644, 1.2812359917653167], [0.6465003541509388, -0.7998770929130626, -0.7727623167052589, -0.385778288483983], [0.8346342537176381, 0.34398818053744207, 0.9958153414776598, 2.479432528149894]]], [[[1.7478473766274505, 0.12441655393012985, -0.6500998515586293, 1.4072615906482988], [-0.6940190793605724, -0.7654364794154525, -0.10100942345862418, 0.2969019336945648], [-0.24168119461517854, 0.279380805505678, -2.6746254338319773, -0.354189152483647], [0.04927453624028841, 0.7242475937078926, 0.3641577383939416, -0.21119356540906664], [2.5104085388288295, 0.9999806218506042, 0.6375816664766865, 0.3313914640665949], [0.4794115165694021, -0.8001515926358916, -1.0052053278775268, 0.24999484923315177], [-1.5738491665960421, 0.6134695820327016, -0.3702804636780358, 0.7263396003394431], [0.13713262321559044, -0.8094256187136643, -0.6870294294854603, -0.39463916176052655]], [[1.3211604023375252, 1.3954188823827685, -0.1709428912111078, 0.15117396876755984], [-0.7797881297660109, 1.429650418295845, 0.2601883128225055, -0.6279731275156257], [-1.292043855148919, 0.5351314314458022, 1.8897226956666475, 0.10749458038354212], [-0.8788115637998645, -0.8536117347524433, -0.5556792812706723, -2.260111761949279], [0.9099362859544923, -0.5607042237555494, -1.8212615775592265, -2.926874734308934], [0.3450079891083276, 0.2146378170453011, 1.4253186459777667, 0.9984395451466545], [-0.40859561737359373, -1.7188009976907643, 0.20790106700523106, -0.5753607617250147], [0.08453218400170862, -1.5889386808139505, -0.2998954517436731, -0.22755072399613424]]]], "dtype": "float64"}, {"__nd__": [[[[-1.283217553200469, -1.7526430510042716, -2.219750228925221, 0.07247970688629665], [1.2008987867033412, -0.040300674122441815, -2.5802582781929226, -0.05143554329586471], [-1.680045755516664, 0.23618083753989433, 1.7267365806986643, 0.3454809186784457], [-2.591899703406305, 1.0434793988167408, 0.3280613657146427, 1.879842226465557], [0.4997894314003217, 0.6902827675656585, -1.7990196782703427, 1.4237289350532232], [-1.2426914349454001, -2.0181746741518585, 0.5033663105602847, 0.7265804556364263], [1.4438488160065102, 1.150761172516862, 0.3670379913146726, 0.32908592176530604], [0.8603412844275949, 0.35934245544530813, 0.45740058267166195, -2.033700302091372]], [[-0.1102357889690413, -0.3684223329941079, 0.5084225616734619, -0.7955745828831363], [0.9359046165693189, 0.7447285531907063, -1.126472963561533, 0.0255204167573806], [0.5436481209762465, -1.0848682607210127, -2.4155594129237383, -0.25002182386202443], [0.24441215765276597, 0.01719907078109228, 1.0841052730677156, 0.29803768471951947], [1.6115491343224655, -0.5703754124625151, 1.6535920401972521, -0.2910756623478765], [0.5009311778232839, -1.88031415221228, -0.7762830078668634, 1.1902472814100071], [0.06678732006881659, 0.010361981123264094, -1.208250543920799, 0.7952855340090841], [-0.9990519988090435, 1.4648137767815526, 0.6153947701106753, -0.5622377920685345]]], [[[-0.8193257105040831, -0.5262561410330757, 0.3781804974056244, 0.2504798117033315], [-1.6195585973912878, 0.3478673535272846, -1.0747138024018115, 1.8406562348247884], [-1.8914342171463479, 0.9392959418065416, -1.1971539817170216, 0.38289396053680635], [1.185055862514284, 0.4724362744593919, -1.178102535057472, -1.5447271859784517], [1.4815179472427271, -0.8105404612870419, 0.7845354017799773, 1.200627724249146], [-1.660945455860599, 1.9807394584952436, -0.12733219017309388, -0.1987749085119995], [-0.9882063638218987, 0.23311442292485554, 0.4838526627432238, 1.0569445116943958], [1.043334803901286, 1.032485255803081, -0.11037300929720532, -0.09188999026702646]], [[0.129518164059734, 0.19163034883291688, 1.2744083673781617, 0.9260583606071836], [-0.1622213991288348, -0.9224324704095341, -1.9160601671917197, -0.2179595190052761], [0.019501536777801212, -1.8780906228528889, -0.3607489942302511, 0.2546367651926883], [0.21745381898090374, -1.4168402641282078, -0.6711909320851933, -1.4620905214302164], [0.8534194478363153, 0.27782843384088657, 2.608417424033557, -0.6026420813766306], [1.5243656802322785, 0.03268630346316836, 0.08564427883978502, 1.0573983354723775], [-1.9373326164042046, -1.197599108410216, 1.5261445982916175, 1.006951591794048], [1.8757092617658273, 0.5658403384579376, -0.47745105185448605, 0.8596386488851683]]], [[[-0.4365011184709154, -0.06746035638836222, -0.12113280095875865, -0.8670384838025453], [2.1249021491611546, 0.7252494729098645, 0.13607437329539981, 0.39601664727769637], [-0.5698494879995372, -0.3717188541281438, 0.12299338861532019, 0.326670260898289], [-2.171913989216002, 1.3202437062671224, -2.05628223217485, -0.9680041181893042], [-1.397643571376839, 0.10635692685900042, -1.7719594310449767, 0.9893237322264471], [0.6910830133349065, 0.9293277485114181, 0.7979755662005868, -0.3330443695391465], [0.24175014787199336, -0.9690763795030434, -0.44009229697547997, 0.4315586828344365], [-1.2734567421587975, -0.3043396931343326, -0.5072780226067014, 0.888058038421329]], [[0.528890499331527, -0.44257780972610183, 0.7964950953781323, 1.93885430526012], [1.0962768560811984, 0.09337967993416428, 0.12725476349539053, -0.9732166847549512], [-0.5435022552752655, 0.2790743940640189, 0.37717155003502356, -1.4315332282300859], [-1.2522229864959438, -1.2332902556568932, -1.160212553212622, -0.6441863247428445], [-0.1684529558712091, -1.1936998964048549, -0.712463864491933, -1.9948708820726033], [-0.25393848499777083, 0.21536522037906206, -0.5748697964764512, -1.5984984164213492], [2.365990606731273, -0.8623559851270041, 0.3870520469481592, -2.2241651510804012], [-0.7924401978499218, 0.09402283601408702, -0.1259334787787657, 0.1044597005573307]]]], "dtype": "float64"}, {"__nd__": [[[[0.0061616249272874015, -0.7984717464967491, -0.9546368728187462, -0.2424338047067513], [-0.7110884069965593, -0.6584661084778122, 0.34872286951146664, 0.16605930694562318], [-0.5205350499838012, 1.016300735850961, -0.13195393030005453, -0.30937438682370055], [0.7609576560230281, 0.18155052651666947, -1.6489514669105452, 0.39829480575756293], [-1.6703998980475099, -0.3820629245817418, 0.22414603061827293, 0.5866102541401338], [-0.9699174947992621, -0.43092833166272587, 0.7094201925603517, 1.947117186213516], [1.9028193196518943, 0.7833452155415035, 1.6065279647331376, -0.5948908742038963], [-0.27957316547319194, 0.7550939388808945, 0.754746632737065, -1.5758415789486626]], [[1.1248247204354844, -0.9019490080486491, -0.15830595377055134, -1.1974649159159896], [-0.39931557395940787, 1.0160383928565953, -1.3385413997300157, -0.679069124575758], [-0.03853380822681345, 0.5473030453334571, 1.6678781533301827, 0.0876980277192774], [1.0137736043315764, -1.5906102166902147, -0.5659031694951211, -2.4218908017297385], [1.1161199166099551, -0.614549808368069, 0.5712592565637973, -1.0931911568073527], [-1.1368608645697462, -1.048568372653241, -0.7498426205113413, -0.34458666755639156], [-1.757562363517266, 0.04141114057346206, 0.5199729630846994, -1.8618753005042779], [-1.5957642578297386, -1.1977493804808463, -0.9671710578901161, 1.6798584824650828]]], [[[0.6902992802034954, -1.3196817472099132, 2.6630247804916247, 0.7475103642859577], [-1.3248031854304643, -1.7433028664838883, 0.3325316897348787, -1.7233795748264051], [0.1028274963884664, 0.8172793622339997, -0.9056351929877333, -0.8009444985876796], [-0.8977340504304309, 1.248950470051646, 0.5956138732810007, 0.4380834983063395], [0.7920514098693098, -0.07773007062688352, -0.6983513261145139, 0.26398984905884176], [0.8653379325929113, -0.2012725992341848, -0.26872261437687783, -0.993010742155404], [1.3529074523710904, 0.7305850872426073, 0.1755749620660547, 0.16556746884512322], [0.35261823369392864, 0.062177038606075194, 0.9667825274241206, 0.17878633725371304]], [[0.7529708740304485, 0.05850669759278116, -0.5802467473706192, 1.1388977852097697], [1.1667719134293832, 1.5389352664965792, -0.42778939043255515, -0.6329684000228946], [-1.8614089462060257, 0.0564744479224794, -0.82687587257564, 1.2008172593568482], [-1.1632015418470414, 0.9568781759741899, -1.236431234612107, -0.11771770071134324], [1.2709680381866277, -0.471494778766695, 0.3901189187426041, 0.7021298225490307], [-1.3497400502203294, 2.259987814657551, -1.3521644604778813, -0.2984661938876075], [0.199184728477823, 0.4110038469854155, 1.0057316187885523, 1.1937477857183], [-1.3408817030876246, -0.168388977934628, -2.079114766315723, 0.27576051570974885]]], [[[-0.6715865593517752, 0.3488396950984449, -1.7127936410170548, -1.128907704418855], [0.49591367181409957, 0.7209325004591691, -0.3258004726050034, -0.45785468376101324], [-0.5625005944024174, -0.5895718692074589, -0.6080636261240562, 2.5658038476410847], [0.7130829855245671, -1.602407177147185, -0.4335849024414317, -1.024545398676068], [0.15823616267436535, 2.1363378080038813, 1.2425088840159777, 0.1443131342317201], [0.40053714301441495, -0.12533962906902008, -1.0109377178488355, -0.5975384187082088], [1.111434234327399, -1.5791316834582236, 1.276073767332804, 2.3408936971975436], [0.07145849398876426, 1.1378807734979153, 0.19331865878684149, -0.27564772282474215]], [[-2.0372428876374236, -0.7284642920570485, 1.5592099876388383, 0.08894914321611781], [-0.3102053345361235, 1.5832280472198963, -0.06300258885462949, -1.960612023977907], [-0.5291662669247973, 1.5230521043139744, 1.1610426723733518, -0.3160720145307965], [0.8403628103288582, -0.0982763676824906, 2.4841860518009993, 0.7708772606825658], [1.2341911487218096, 0.5275670751156154, 1.1437045473371366, 0.7432397248004905], [0.1699018552063201, -0.374822966388521, -0.267007583288147, -0.04444134802435802], [0.854521462232738, 0.7438057004228797, 1.489354201562345, -0.08204462397248176], [0.2815809496586738, -2.010912862442713, -0.6208733073846089, -0.5788090526246877]]]], "dtype": "float64"}, {"__nd__": [[0, 0, 0, 0, 1, 1, 1, -1], [2, 3, 3, 3, 3, 4, 4, 4], [9, 9, 9, 9, 9, 9, 9, 9]], "dtype": "int64"}], "e": {"__tup__": [{"__nd__": [[[[0.0061616249272874015, -0.7984717464967491, -0.9546368728187462, -0.2424338047067513], [-0.6252478824685176, -0.6752219911819132, 0.19273670157744055, 0.1171708281591144], [-0.4773224802608816, 0.8355471434107538, -0.19694902429101152, -0.29705371575999223], [-0.5872392256856876, -0.05897597995841176, 0.10072114149384136, 0.006442981202262517], [-1.6703998980475099, -0.3820629245817418, 0.22414603061827293, 0.5866102541401338], [-1.3018789685519825, -0.4077708155353988, 0.47944678391311335, 1.3023673907474693], [-0.9404821027754616, -0.41088279543129724, 0.7127375430208182, 1.8774192113088197], [0.0, 0.0, 0.0, 0.0]], [[1.1248247204354844, -0.9019490080486491, -0.15830595377055134, -1.1974649159159896], [0.4489712983500326, -0.051450957976930625, -0.6816607708774237, -0.9675913466255133], [0.05077113765769819, 0.4430013386934294, 0.24063635086074198, -0.43075482746468313], [0.45691117825162647, -0.4620844744876032, -0.7587462052237683, -1.5565989538039], [1.1161199166099551, -0.614549808368069, 0.5712592565637973, -1.0931911568073527], [-0.8363895664923603, -0.9906850021026216, -0.5736523932296926, -0.44442514132890876], [-1.2940416821479148, -0.3173989524973566, 0.1865038366210209, -1.377373249917539], [0.0, 0.0, 0.0, 0.0]]], [[[0.6902992802034954, -1.3196817472099132, 2.6630247804916247, 0.7475103642859577], [-1.3248031854304643, -1.7433028664838883, 0.3325316897348787, -1.7233795748264051], [-0.9253507737258098, -1.026849543047143, -0.013908572228552, -1.4652813702017589], [-0.6307917477826883, -0.2885244433518096, -0.2096194285206144, -1.0970936988001354], [-0.6409773818473306, -0.6389560280551463, -0.19415052630943044, -1.2112417768550958], [0.8653379325929113, -0.2012725992341848, -0.26872261437687783, -0.993010742155404], [1.002919255986116, 0.06167700798887273, -0.14335167532551543, -0.6660856226432106], [1.1361993707851066, 0.5119428646790487, 0.24587470314341062, 0.0264989958235971]], [[0.7529708740304485, 0.05850669759278116, -0.5802467473706192, 1.1388977852097697], [1.1667719134293832, 1.5389352664965792, -0.42778939043255515, -0.6329684000228946], [-1.3969987200392253, 0.2838287559872589, -0.7656708613747907, 0.9195827980451428], [-0.9593905562087723, 0.6445069914722292, -0.8394942794747619, 0.4197503164378737], [0.5560246882694362, 0.1974105859395384, -0.13703244088261965, 0.37940572858721006], [-1.3497400502203294, 2.259987814657551, -1.3521644604778813, -0.2984661938876075], [-0.9839344569021952, 1.8233180120745611, -0.7953061894238764, 0.053946155423596896], [-1.077935066133761, 1.0788311193693247, -1.2007589908785963, 0.1638284906233397]]], [[[-0.6715865593517752, 0.3488396950984449, -1.7127936410170548, -1.128907704418855], [-0.36164464962024784, 0.4476209643541245, -1.3445818840284292, -0.9507600268410067], [-0.37428613231566077, 0.17138194783935937, -1.086842431490026, 0.05789270493835317], [0.07378144005137029, -0.23556544249256275, -0.7353605389399177, -0.18041721817044382], [0.4526774297469324, 0.6913534713599029, -0.35006410919026787, -0.4132757759870316], [0.09190408111872579, 0.4981631633602183, -0.16772228446347595, 0.008485158193101031], [0.3562606366012441, -0.11334928722575104, 0.08366607779750633, -0.1964736304724151], [0.2471326930681546, -0.025096659985732416, -0.02427525796128634, 0.32861134640224504]], [[-2.0372428876374236, -0.7284642920570485, 1.5592099876388383, 0.08894914321611781], [-0.7783596681964413, 0.9565892200684748, 0.37673645818573404, -1.4050300444194688], [-1.0533983963376672, 0.7048077990561311, 1.1438808321796212, -0.386811028209297], [0.2372696264399731, 0.628978162207989, 1.7505770794124917, 0.09684857526858728], [0.9279638946232467, 0.6351779014542905, 1.1572397902032856, 0.4150933526719776], [-1.430237247382717, -0.19329309818908208, 1.275692953646345, -0.15316081870987006], [0.5651819336485563, 0.41843404831571257, 1.3680725144639583, 0.2555078922455309], [0.48142295470446256, 0.2482963523288267, 1.2326202362306533, 0.14475638884606987]]]], "dtype": "float64"}, {"__nd__": [[0, 1, 2, 3, 0, 1, 2, -1], [0, 0, 1, 2, 3, 0, 1, 2], [0, 1, 2, 3, 4, 5, 6, 7]], "dtype": "int64"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(packed_causal_attention, json.loads(_TESTS), 'torch', *[1e-06, 1e-08])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")